# 1.3 Création des bases assurés et sinistrés


In [6]:
import numpy as np, pandas as pd

In [7]:
df_trainpol = pd.read_csv("trainpol_propre.csv", sep=',')
df_trainclaim = pd.read_csv("trainclaim_propre.csv", sep=',')
#df_baseclimat = pd.read_csv("baseclimat_propre.csv", dec=',')

### Création de la base des assurés

Nous allons utiliser la fusion en utilisant left pour faire apparaitre dans notre base de données définitive les assuré sinitré comme ceux non sinistrés.

In [8]:
base_ass = pd.merge(df_trainpol, df_trainclaim, on = ['id_client', 'id_vehicle', 'id_year'],how = 'left')
base_ass.head()

,Unnamed: 0_x,id_client,id_vehicle,id_policy,id_year,pol_bonus,pol_coverage,pol_duration,pol_sit_duration,pol_pay_freq,...,vh_sale_begin,vh_sale_end,vh_speed,vh_type,vh_value,vh_weight,Unnamed: 0_y,id_claim,claim_nb,claim_amount
0,1,A00036953,V01,A00036953-V01,Year 0,0.50,4,11.0,1.0,Yearly,...,2.0,1.0,216.0,Tourism,32000.0,1399,NaN,NaN,NaN,NaN
1,3,A00018024,V01,A00018024-V01,Year 0,0.64,3,5.0,1.0,Yearly,...,18.0,15.0,190.0,Tourism,12517.0,860,NaN,NaN,NaN,NaN
2,6,A00071159,V01,A00071159-V01,Year 0,0.50,4,9.0,1.0,Biannual,...,7.0,6.0,208.0,Tourism,29200.0,1580,NaN,NaN,NaN,NaN
3,9,A00063340,V01,A00063340-V01,Year 0,0.50,4,2.0,2.0,Yearly,...,9.0,5.0,188.0,Tourism,38850.0,1782,NaN,NaN,NaN,NaN
4,14,A00011762,V01,A00011762-V01,Year 0,0.50,4,10.0,6.0,Biannual,...,11.0,6.0,195.0,Tourism,21435.0,1260,NaN,NaN,NaN,NaN


### Création de la base des assurés sinistrés

Cette fusion va nous permettre d'obtenir une base donnée qui contient que les assurés qui ont eu un sinistre

In [9]:
Base_sin = pd.merge(df_trainpol, df_trainclaim, on = ['id_client', 'id_vehicle', 'id_year'],how = 'inner')
Base_sin.head()

,Unnamed: 0_x,id_client,id_vehicle,id_policy,id_year,pol_bonus,pol_coverage,pol_duration,pol_sit_duration,pol_pay_freq,...,vh_sale_begin,vh_sale_end,vh_speed,vh_type,vh_value,vh_weight,Unnamed: 0_y,id_claim,claim_nb,claim_amount
0,37,A00031330,V01,A00031330-V01,Year 0,0.50,3,30.0,1.0,Yearly,...,13.0,8.0,170.0,Tourism,11950.0,950,6751,CL01,1,-338.76
1,56,A00056658,V01,A00056658-V01,Year 0,0.50,4,13.0,1.0,Biannual,...,2.0,1.0,180.0,Tourism,23100.0,1385,13149,CL01,1,0.00
2,62,A00004603,V02,A00004603-V02,Year 0,0.50,4,12.0,1.0,Quarterly,...,2.0,1.0,195.0,Tourism,29200.0,1581,481,CL01,1,130.11
3,68,A00034712,V01,A00034712-V01,Year 0,0.62,2,4.0,2.0,Monthly,...,26.0,23.0,190.0,Tourism,30295.0,1370,7499,CL01,1,230.72
4,77,A00081062,V02,A00081062-V02,Year 0,0.50,3,19.0,4.0,Yearly,...,19.0,17.0,161.0,Tourism,14788.0,1005,9626,CL01,1,1236.00


### Fusion de base des assurés et des assurés sinitrés avec la base climatique

In [10]:
#Base_sin = pd.merge(Base_sin, pg_claim, on = ['id_client', 'id_vehicle', 'id_year'],how = 'inner')
#Base_sin.head()

In [11]:
print("Les dimensions de notre base des assurés et des non assurés est :",base_ass.shape)

Les dimensions de notre base des assurés et des non assurés est : (32895, 36)


On constate que notre base totale (contenant les sinistrés et non sinistré) est composé de 32895 observations et 34 variables

In [12]:
print("Les dimensions de notre base des assurés uniquement est :",Base_sin.shape)

Les dimensions de notre base des assurés uniquement est : (4968, 36)


On constate que notre base totale (contenant les sinistrés et non sinistré) est composé de 4968 observations et 34 variables

#### Hypothèse : Nous allons considérer les individus ayant un obtenu un sinistre et dont le montant de sinistre est egale à zéro comme des individus n'ayant pas eu de sinistre tout au long de l'année car nous avons pas assez d'informations pour pouvoir traiter ce type de situation.

Sur la base de cette hypothèse dans notre base totale, nous allons modifier les observations suivantes : si la variable (claim_amount) est égale à 0 mettre la variable (claim_nb) est égale à 0

In [13]:
# Modifier claim_nb lorsque claim_amount est égal à 0
base_ass.loc[base_ass["claim_amount"] == 0, "claim_nb"] = 0

Sur la base de cette hypothèse dans notre base sinistre, nous allons supprimer les observations suivantes : si la variable (claim_nb) est égale à 1 et la variable (claim_amount) est égale à 0

In [14]:
# Filtrer et supprimer les observations où claim_nb == 1 et claim_amount == 0
Base_sin = Base_sin[~((Base_sin["claim_nb"] == 1) & (Base_sin["claim_amount"] == 0))]

Nous allons essayer de de calculer la frequence d'exposition de au sinistre dans notre base de donnée de sinistre pour ce faire, nous allons considérer que l'exposition pour une année de l'ensemble de nos assurés est de 1 ensuite considérer que les individus qui n'ont eu aucun sinistre dans la base de données Base_1 sont ceux qui sont NaN au niveau de variable Claim_nb.

In [15]:
base_ass["id_year"] = base_ass["id_year"].replace("Year 0", 1)

C:\Users\mnmor\AppData\Local\Temp\ipykernel_4444\317547394.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  base_ass["id_year"] = base_ass["id_year"].replace("Year 0", 1)


In [16]:
base_ass["claim_nb"] = base_ass["claim_nb"].replace(np.nan, 0)

In [17]:
base_ass["claim_amount"] = base_ass["claim_amount"].replace(np.nan, 0)

Déterminons la fréquence d'être sinistrer au sein de notre base de donnée. 

In [18]:
freq = sum(base_ass.claim_nb)/sum(base_ass.id_year)
freq

0.1443684450524396

La fréquence de sinistre de notre bas de données est d'environ 14 %

Déterminons le cout moyen de sinistre 

In [19]:
cm = sum(base_ass.claim_amount) / sum(base_ass.claim_nb)
cm

909.8515266371869

Le cout moyen de sinistre est de 909.8515 euros environ

Déterminons le montant d'une eventuelle prime 

In [20]:
pp = freq*cm
pp

131.353850129199

La prime de sinistre basique est d'environ 131.3538 euros 

In [21]:
base_ass.claim_nb.value_counts(dropna=False)

claim_nb
0.0    28146
1.0     4749
Name: count, dtype: int64

In [22]:
base_ass.claim_amount.value_counts(dropna=False)

claim_amount
0.00       28146
1236.00      258
106.14       109
1010.41       92
110.98        44
           ...  
792.19         1
100.36         1
539.21         1
2472.96        1
5013.73        1
Name: count, Length: 3829, dtype: int64

### Traitement de la base définitive

In [23]:
base_ass.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32895 entries, 0 to 32894
Data columns (total 36 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0_x      32895 non-null  int64  
 1   id_client         32895 non-null  object 
 2   id_vehicle        32895 non-null  object 
 3   id_policy         32895 non-null  object 
 4   id_year           32895 non-null  int64  
 5   pol_bonus         32895 non-null  float64
 6   pol_coverage      32895 non-null  int64  
 7   pol_duration      32895 non-null  float64
 8   pol_sit_duration  32895 non-null  float64
 9   pol_pay_freq      32895 non-null  object 
 10  pol_payd          32895 non-null  object 
 11  pol_usage         32895 non-null  object 
 12  pol_insee_code    32895 non-null  object 
 13  drv_drv2          32895 non-null  object 
 14  drv_age1          32895 non-null  float64
 15  drv_age2          32895 non-null  float64
 16  drv_sex1          32895 non-null  object

In [24]:
Base_sin.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4749 entries, 0 to 4967
Data columns (total 36 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0_x      4749 non-null   int64  
 1   id_client         4749 non-null   object 
 2   id_vehicle        4749 non-null   object 
 3   id_policy         4749 non-null   object 
 4   id_year           4749 non-null   object 
 5   pol_bonus         4749 non-null   float64
 6   pol_coverage      4749 non-null   int64  
 7   pol_duration      4749 non-null   float64
 8   pol_sit_duration  4749 non-null   float64
 9   pol_pay_freq      4749 non-null   object 
 10  pol_payd          4749 non-null   object 
 11  pol_usage         4749 non-null   object 
 12  pol_insee_code    4749 non-null   object 
 13  drv_drv2          4749 non-null   object 
 14  drv_age1          4749 non-null   float64
 15  drv_age2          4749 non-null   float64
 16  drv_sex1          4749 non-null   object 
 17  

In [25]:
df = base_ass.copy()

In [26]:
print("La nouvelle dimension de notre base définitive est :",df.shape)

La nouvelle dimension de notre base définitive est : (32895, 36)


In [27]:
len(df)

32895

### Gestion des doublons de la base définitive

In [28]:
#pour voir directement combien d'observation on a en double
print("La base contient", len(df)-df.id_client.nunique(), "doublons") # la difference entre le nombre d'observation total et le nombre d'observation sans doublons

La base contient 2621 doublons


In [29]:
#pour compter les doublon pur/parfaits
print("La base contient",sum(df.duplicated()),"Doublons purs")

La base contient 0 Doublons purs


In [30]:
# Affichons les doubons impurs
doublons_df = df[df.duplicated('id_client', keep=False)].sort_values("id_client")
doublons_df

,Unnamed: 0_x,id_client,id_vehicle,id_policy,id_year,pol_bonus,pol_coverage,pol_duration,pol_sit_duration,pol_pay_freq,...,vh_sale_begin,vh_sale_end,vh_speed,vh_type,vh_value,vh_weight,Unnamed: 0_y,id_claim,claim_nb,claim_amount
21548,65816,A00000052,V01,A00000052-V01,1,0.54,1,5.0,2.0,Yearly,...,28.0,18.0,155.0,Tourism,12867.0,880,NaN,NaN,0.0,0.00
14786,45136,A00000052,V02,A00000052-V02,1,0.54,4,2.0,2.0,Yearly,...,11.0,10.0,219.0,Tourism,37046.0,1575,NaN,NaN,0.0,0.00
12660,38612,A00000103,V01,A00000103-V01,1,0.50,4,1.0,1.0,Yearly,...,4.0,1.0,182.0,Tourism,37330.0,1604,3790.0,CL02,1.0,419.63
12659,38612,A00000103,V01,A00000103-V01,1,0.50,4,1.0,1.0,Yearly,...,4.0,1.0,182.0,Tourism,37330.0,1604,292.0,CL01,1.0,1389.39
14674,44801,A00000221,V01,A00000221-V01,1,0.50,4,3.0,2.0,Biannual,...,13.0,9.0,203.0,Tourism,27210.0,1342,3931.0,CL01,1.0,671.47
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1173,3533,A00091339,V01,A00091339-V01,1,0.50,4,27.0,1.0,Biannual,...,8.0,5.0,205.0,Tourism,27300.0,1390,NaN,NaN,0.0,0.00
29991,91160,A00091344,V01,A00091344-V01,1,0.50,4,4.0,4.0,Yearly,...,6.0,5.0,178.0,Tourism,24100.0,1365,9343.0,CL02,1.0,356.26
29990,91160,A00091344,V01,A00091344-V01,1,0.50,4,4.0,4.0,Yearly,...,6.0,5.0,178.0,Tourism,24100.0,1365,4218.0,CL01,1.0,471.93
27984,85008,A00091466,V01,A00091466-V01,1,0.50,4,25.0,2.0,Monthly,...,3.0,1.0,160.0,Tourism,12690.0,975,10849.0,CL01,1.0,856.64


Après analyse des doublons sur la base du numéro d'identification du client, on constate que les doublons ne sont pas forcémment les doublons car ils s'agit d'observations qui ne diffrèrent pas d'une variable à une autre sauf pour le cas de la variable "id_claim" suivant le même type du numéro d'identification du client.

### Gestion des données manquantes dans la base finale

In [31]:
#première vérification
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32895 entries, 0 to 32894
Data columns (total 36 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0_x      32895 non-null  int64  
 1   id_client         32895 non-null  object 
 2   id_vehicle        32895 non-null  object 
 3   id_policy         32895 non-null  object 
 4   id_year           32895 non-null  int64  
 5   pol_bonus         32895 non-null  float64
 6   pol_coverage      32895 non-null  int64  
 7   pol_duration      32895 non-null  float64
 8   pol_sit_duration  32895 non-null  float64
 9   pol_pay_freq      32895 non-null  object 
 10  pol_payd          32895 non-null  object 
 11  pol_usage         32895 non-null  object 
 12  pol_insee_code    32895 non-null  object 
 13  drv_drv2          32895 non-null  object 
 14  drv_age1          32895 non-null  float64
 15  drv_age2          32895 non-null  float64
 16  drv_sex1          32895 non-null  object

In [32]:
# Deuxième vérification
df.isna().sum()/len(df)*100

Unnamed: 0_x         0.000000
id_client            0.000000
id_vehicle           0.000000
id_policy            0.000000
id_year              0.000000
pol_bonus            0.000000
pol_coverage         0.000000
pol_duration         0.000000
pol_sit_duration     0.000000
pol_pay_freq         0.000000
pol_payd             0.000000
pol_usage            0.000000
pol_insee_code       0.000000
drv_drv2             0.000000
drv_age1             0.000000
drv_age2             0.000000
drv_sex1             0.000000
drv_sex2             0.000000
drv_age_lic1         0.000000
drv_age_lic2         0.000000
vh_age               0.000000
vh_cyl               0.000000
vh_din               0.000000
vh_fuel              0.000000
vh_make              0.000000
vh_model             0.000000
vh_sale_begin        0.000000
vh_sale_end          0.000000
vh_speed             0.000000
vh_type              0.000000
vh_value             0.000000
vh_weight            0.000000
Unnamed: 0_y        84.897401
id_claim  

### Aggrégation de nos variables sur notre base sinistre

In [33]:
# Liste des colonnes que nous allons supprimer 'id_claim', 'claim_amount', 'claim_nb', 'id_policy', 'id_year' sauf 'id_client' qui sera utilisée comme identifiant plutard
other_columns = [col for col in df.columns if col not in ['id_client', 'id_claim', 'claim_amount', 'claim_nb', 'id_policy', 'id_year']] 

# Grouper par id_client et agréger les colonnes
agg_sin = Base_sin.groupby('id_client').agg(
    nb_sin=pd.NamedAgg(column='id_claim', aggfunc='nunique'),  # Compter le nombre de sinistre par client
    remb_sin=pd.NamedAgg(column='claim_amount', aggfunc='sum'),  # Somme des montants des réclamations par client
    **{col: pd.NamedAgg(column=col, aggfunc='first') for col in other_columns}  # Garder les autres colonnes
).reset_index()

# Afficher les premières lignes du DataFrame résultant
agg_sin

,id_client,nb_sin,remb_sin,Unnamed: 0_x,id_vehicle,pol_bonus,pol_coverage,pol_duration,pol_sit_duration,pol_pay_freq,...,vh_fuel,vh_make,vh_model,vh_sale_begin,vh_sale_end,vh_speed,vh_type,vh_value,vh_weight,Unnamed: 0_y
0,A00000016,1,555.48,9946,V01,0.50,4,6.0,2.0,Monthly,...,Diesel,OPEL,CORSA,5.0,4.0,163.0,Tourism,14900.0,1141,7470
1,A00000056,1,1236.00,13158,V01,0.64,3,2.0,2.0,Monthly,...,Diesel,FORD,FOCUS,12.0,9.0,184.0,Tourism,18320.0,1277,501
2,A00000070,1,158.28,77138,V01,0.50,4,1.0,1.0,Biannual,...,Gasoline,RENAULT,TWINGO,3.0,1.0,169.0,Tourism,10150.0,955,2711
3,A00000092,1,1429.82,84686,V01,0.50,4,16.0,2.0,Monthly,...,Diesel,CITROEN,PASSAT,11.0,7.0,208.0,Tourism,25500.0,1375,2759
4,A00000103,2,1809.02,38612,V01,0.50,4,1.0,1.0,Yearly,...,Diesel,VOLKSWAGEN,TIGUAN,4.0,1.0,182.0,Tourism,37330.0,1604,292
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4188,A00091354,1,101.59,7233,V01,0.50,4,27.0,2.0,Monthly,...,Diesel,CITROEN,SCENIC,6.0,2.0,185.0,Tourism,26000.0,1430,12550
4189,A00091372,1,1325.70,32017,V01,0.50,4,1.0,1.0,Monthly,...,Gasoline,PEUGEOT,MEGANE,9.0,6.0,192.0,Tourism,17350.0,1175,2021
4190,A00091459,1,310.79,50921,V01,0.51,4,6.0,2.0,Yearly,...,Diesel,PEUGEOT,POLO,3.0,3.0,164.0,Tourism,12490.0,1089,14292
4191,A00091466,2,1474.64,85008,V01,0.50,4,25.0,2.0,Monthly,...,Gasoline,OPEL,AGILA,3.0,1.0,160.0,Tourism,12690.0,975,10849


In [34]:
#pour voir directement combien d'observation on a en double
print("La base agrégée des sinistrés contient", len(agg_sin)-agg_sin.id_client.nunique(), "doublons")

La base agrégée des sinistrés contient 0 doublons


In [35]:
#pour compter les doublon pur/parfaits
print("La base agrégée des sinistrés contient",sum(agg_sin.duplicated()),"doublons purs")

La base agrégée des sinistrés contient 0 doublons purs


In [36]:
print("La nouvelle dimension de notre base agrégée des sinistrés est :",agg_sin.shape)

La nouvelle dimension de notre base agrégée des sinistrés est : (4193, 33)


In [37]:
agg_sin.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4193 entries, 0 to 4192
Data columns (total 33 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id_client         4193 non-null   object 
 1   nb_sin            4193 non-null   int64  
 2   remb_sin          4193 non-null   float64
 3   Unnamed: 0_x      4193 non-null   int64  
 4   id_vehicle        4193 non-null   object 
 5   pol_bonus         4193 non-null   float64
 6   pol_coverage      4193 non-null   int64  
 7   pol_duration      4193 non-null   float64
 8   pol_sit_duration  4193 non-null   float64
 9   pol_pay_freq      4193 non-null   object 
 10  pol_payd          4193 non-null   object 
 11  pol_usage         4193 non-null   object 
 12  pol_insee_code    4193 non-null   object 
 13  drv_drv2          4193 non-null   object 
 14  drv_age1          4193 non-null   float64
 15  drv_age2          4193 non-null   float64
 16  drv_sex1          4193 non-null   object 
